# Stress Prediction V26 — Conservative Direction Extrapolation
기존 최고 예측과 성능이 낮았던 두 독립 모델의 공통 이동 방향을 이용한 보수적 외삽 후보입니다.

**주의:** 로컬 CV 기반 신규 모델이 아니라 Public 제출 방향을 이용한 후보이므로 5% 후보 하나만 먼저 확인합니다.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# V26 — Conservative Prediction-Direction Extrapolation
# 모델을 Test로 재학습하지 않습니다. 이미 정상적으로 생성된 팀 내부 모델의
# 예측 방향만 결합하며, 원본 Test 통계나 정답 정보는 사용하지 않습니다.

ROOT = Path("/Users/bitsaem/Desktop/stress_project")
TARGET = "stress_score"

best = pd.read_csv(ROOT / "submit_team_best_original_pair_neighbor.csv")
v14 = pd.read_csv(ROOT / "submit_v14_blend95.csv")
v10 = pd.read_csv(ROOT / "submit_v10_v9_age_ensemble.csv")

for name, frame in {"best": best, "v14": v14, "v10": v10}.items():
    if not frame["ID"].equals(best["ID"]):
        raise ValueError(f"{name}의 ID 순서가 기존 최고 제출과 다릅니다.")

base = best[TARGET].to_numpy(float)
weaker_consensus = (
    v14[TARGET].to_numpy(float) + v10[TARGET].to_numpy(float)
) / 2.0
direction = base - weaker_consensus

print("기존 최고와 약한 모델 합의 예측의 평균 절대 차이:", np.mean(np.abs(direction)))
print("방향이 0이 아닌 행:", int(np.sum(np.abs(direction) > 1e-12)), "/", len(base))

rows = []
for strength in (0.05, 0.10, 0.20):
    prediction = np.clip(base + strength * direction, 0.0, 1.0)
    output = best.copy()
    output[TARGET] = prediction
    label = int(strength * 100)
    path = ROOT / f"submit_v26_direction_{label:02d}.csv"
    output.to_csv(path, index=False, encoding="utf-8")
    rows.append({
        "candidate": f"direction_{label:02d}",
        "strength": strength,
        "mean_abs_change": np.mean(np.abs(prediction - base)),
        "max_abs_change": np.max(np.abs(prediction - base)),
        "changed_rows": int(np.sum(np.abs(prediction - base) > 1e-12)),
        "path": str(path),
    })

summary = pd.DataFrame(rows)
display(summary)
print("권장 제출 순서: direction_05 한 개만 먼저 제출 → 악화 시 즉시 중단")
print("주의: V26은 Public 방향 기반 후보라 로컬 CV MAE를 계산할 수 없습니다.")



기존 최고와 약한 모델 합의 예측의 평균 절대 차이: 0.01378852170833334
방향이 0이 아닌 행: 1539 / 3000


,candidate,strength,mean_abs_change,max_abs_change,changed_rows,path
0,direction_05,0.05,0.000689,0.010475,1537,/Users/bitsaem/Desktop/stress_project/submit_v...
1,direction_10,0.10,0.001378,0.020950,1537,/Users/bitsaem/Desktop/stress_project/submit_v...
2,direction_20,0.20,0.002754,0.041900,1537,/Users/bitsaem/Desktop/stress_project/submit_v...


권장 제출 순서: direction_05 한 개만 먼저 제출 → 악화 시 즉시 중단
주의: V26은 Public 방향 기반 후보라 로컬 CV MAE를 계산할 수 없습니다.
